# Run GELOS embedding generation pipeline interactively

In [ ]:
from pathlib import Path
import torch
from gelos.gelosdatamodule import GELOSDataModule
import yaml
from lightning.pytorch import Trainer
from tqdm import tqdm
import geopandas as gpd
from terratorch.tasks import EmbeddingGenerationTask
from gelos.generation import LenientEmbeddingGenerationTask, setup_embedding_run
from lightning.pytorch.cli import instantiate_class
import numpy as np
from matplotlib import transforms, patches
from matplotlib.colors import ListedColormap
import matplotlib.pyplot as plt

In [ ]:
yaml_path = Path("/app/configs/exp007_terramind.yaml")
raw_data_dir = Path("/app/data/raw")
embedding_dir = Path("/app/data/interim")
with open(yaml_path, "r") as f:
    yaml_config = yaml.safe_load(f)
version = yaml_config['data_version']
figures_dir = Path("/app/reports/figures")

## Set up the embedding run

In [ ]:
datamodule, task, trainer, output_dir = setup_embedding_run(
    yaml_path=yaml_path,
    raw_data_dir=raw_data_dir,
    embedding_dir=embedding_dir,
)

## Run embedding generation step by step

### Inspect outputs of dataset

In [ ]:
datamodule.setup(stage="predict")

In [ ]:
print(f"Dataset sensors and bands: {datamodule.dataset.bands}")

In [ ]:
for k, v in datamodule.dataset[0].items():
    if k == "image" and isinstance(v, dict):
        for sensor, data in v.items():
            print(sensor, data.shape)
    elif k == "image":
        print (k, v.shape)
    else:
        print(k, v)

In [ ]:
plot = datamodule.dataset.plot(datamodule.dataset[0])

## Generate Embeddings

### Generate one embedding and inspect

In [ ]:
sample = datamodule.dataset[0] # get one sample from the dataset
# image = sample["image"].unsqueeze(0) # add dummy batch dimension
image = sample["image"]
for k, v in image.items():
    print(k, v.shape)
    if len(v.shape) < 5:
        image[k] = v.unsqueeze(0)
        print(v.shape) # add dummy batch dimension
final_layer_embedding = task.model(image)[-1]

In [ ]:
final_layer_embedding.shape

#### Visualize embeddings and slicing

This section is not necessary for the embedding pipeline, but is a useful check on what patches are being sliced using the slicing args.

Currently set up for Prithvi EO 300M embedding shapes.

In [ ]:
# Customize visualization paramters
vis_embed_depth = 128
n_per_grid = 36
date_ranges = ["Jan-Mar", "Apr-Jun", "Jul-Sep", "Oct-Dec"]
has_cls = True

In [ ]:
from src.plot_embeddings import plot_embeddings

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
plot_embeddings(
    final_layer_embedding = final_layer_embedding,
    yaml_config = yaml_config,
    version = version,
    figures_dir = None,
)

### Generate all embeddings

In [ ]:
# Generate Embeddings
trainer.predict(model=task, datamodule=datamodule)